# Kp-Vsys — figures finales

Notebook de production quotidien : génère toutes les figures publiables à partir
des fichiers NPZ de grille logL pré-calculés sur le cluster.

**Contenu**
1. Chemins et paramètres
2. Alpha_frac multi-visites
3. Cartes Kp-vsys par visite + contours superposés
4. Carte combinée Kp-vsys
5. Diagnostic par ordre spectral
6. Cartes alpha–Kp et alpha–vsys
7. Sauvegarde des posteriors (pour comparaison de modèles)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

import starships.logl_grid as lg
from starships.plotting_fcts import (
    plot_kpvsys_map,
    plot_contours_overlay,
    plot_logl_per_order,
    plot_alpha_rv_map,
    plot_loo_contributions,
    find_isolated_peak_sigma,
)

## 1. Chemins et paramètres

In [ ]:
path_results = Path.home() / 'scratch/DataAnalysis/SPIRou/logl_grids/'

# Pattern glob — encode les paramètres de grille dans le nom de fichier
file_glob = 'take3_HRR_modif_disso_31254924_all_kp*_visit*.npz'

# Position attendue du pic (pour les lignes de référence)
kp_ref      = 227.15  # km/s
rv_expected = 0.0     # km/s

# Grille alpha pour la marginalisation
alpha_array = np.linspace(0.01, 2., 31)

files = sorted(path_results.glob(file_glob))
print(f'{len(files)} fichier(s) trouvé(s) :')
for f in files:
    print(' ', f.name)

## 2. Alpha_frac multi-visites

`alpha_frac` est la fraction du signal planétaire reçue lors de chaque exposition.
Les expositions avec `alpha_frac > 0.5` sont celles qui contribuent au signal détecté.

In [ ]:
%matplotlib inline

# Palette Paul Tol (colorblind-safe)
_COLORS = ['#4477AA', '#EE6677', '#228833', '#CCBB44', '#66CCEE', '#AA3377']

fig, ax = plt.subplots(figsize=(9, 3))
for i_file, filename in enumerate(files):
    lg.load_logl_results([filename])
    af   = lg._loaded_extra['alpha_frac']
    ph   = lg._loaded_extra.get('phase', np.arange(len(af)))
    color = _COLORS[i_file % len(_COLORS)]
    ax.plot(ph, af, 'o-', markersize=5, color=color, lw=1.2, label=f'Visite {i_file + 1}')

ax.axhline(0.5, linestyle='--', color='gray', lw=0.9, label='seuil signal')
ax.set_xlabel('Phase orbitale', fontsize=13)
ax.set_ylabel(r'$\alpha_\mathrm{frac}$', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()

# Recharger toutes les visites
lg.load_logl_results(files)

In [ ]:
# Indices des exposures avec signal (toutes visites confondues)
alpha_frac = lg._loaded_extra['alpha_frac']
(idx_signal,) = np.nonzero(alpha_frac > 0.5)
print(f'Exposures avec signal planétaire : {len(idx_signal)} / {len(alpha_frac)}')

## 3. Cartes Kp-vsys par visite

### 3a. Carte individuelle par visite

In [ ]:
# Stockage des posteriors pour la superposition
visit_posteriors = []
visit_vsys_axes  = []
visit_kp_axes    = []
visit_labels     = []

for i_file, filename in enumerate(files, start=1):
    lg.load_logl_results([filename])

    af_v = lg._loaded_extra['alpha_frac']
    (idx_v,) = np.nonzero(af_v > 0.5)

    post_v, vsys_v, kp_v, marg_vsys_v, marg_kp_v = lg.compute_kpvsys_posterior(
        alpha_array=alpha_array, idx_signal=idx_v, oversample=2,
    )
    visit_posteriors.append(post_v)
    visit_vsys_axes.append(vsys_v)
    visit_kp_axes.append(kp_v)
    visit_labels.append(f'Visite {i_file}')

    fig, axes = plot_kpvsys_map(
        post_v, vsys_v, kp_v, marg_vsys_v, marg_kp_v,
        n_sigma=3, sigma_display='contours',
    )
    axes[0].axvline(rv_expected, color='r', linestyle='--', alpha=0.5)
    axes[0].axhline(kp_ref,      color='r', linestyle='--', alpha=0.5)
    fig.suptitle(f'Visite {i_file} : {filename.name}', y=1.02, fontsize=11)
    plt.show()

lg.load_logl_results(files)  # recharger toutes visites

### 3b. Contours superposés (toutes visites)

In [ ]:
fig, ax = plot_contours_overlay(
    visit_posteriors, visit_vsys_axes, visit_kp_axes,
    labels=visit_labels,
    n_sigma=1,                  # contour à 1σ par visite
    # sigma_levels=[1., 2.],    # niveaux personnalisés
    # vsys_lim=(-30, 30),
    # kp_lim=(150, 300),
    rv_expected=rv_expected,
    kp_ref=kp_ref,
    figsize=(6, 5),
)
ax.set_title('Contours 1σ par visite', fontsize=12)
plt.show()

## 4. Carte combinée Kp-vsys

Toutes les visites sont combinées pour maximiser le rapport signal/bruit.

In [ ]:
posterior, vsys_os, kp_os, margin_vsys, margin_kp = lg.compute_kpvsys_posterior(
    alpha_array=alpha_array,
    idx_signal=idx_signal,
    oversample=2,
    # kind='G',   # décommenter pour Gibson logL
)

sigma_isolated = find_isolated_peak_sigma(posterior, vsys_os, kp_os)
print(f'Sigma max (pic isolé) : {sigma_isolated:.2f}σ')

fig, axes = plot_kpvsys_map(
    posterior, vsys_os, kp_os, margin_vsys, margin_kp,
    n_sigma=3, sigma_display='contours',
    # sigma_levels='auto',      # = sigma_isolated calculé automatiquement
    # vsys_lim=(-30, 30),
    # kp_lim=(150, 300),
)
axes[0].axvline(rv_expected, color='r', linestyle='--', alpha=0.5, label='vsys ref')
axes[0].axhline(kp_ref,      color='r', linestyle='--', alpha=0.5)
axes[0].legend(fontsize=10)

max_ind = np.unravel_index(np.argmax(posterior), posterior.shape)
print(f'Pic : vsys = {vsys_os[max_ind[0]]:.2f} km/s,  Kp = {kp_os[max_ind[1]]:.2f} km/s')

## 5. Diagnostic par ordre spectral

Chaque panneau montre la CCF (ou logL) intégrée sur les exposures de signal pour un seul ordre.
Une détection réelle devrait apparaître au même (vsys, Kp) dans plusieurs ordres.

In [ ]:
# CCF par ordre : somme sur les exposures de signal, axe des ordres conservé
# shape : (n_vsys, n_kp, n_orders)
ccf_orders = lg.get_ccf(idx_exposure=idx_signal, sum_axis=-2, kind='BL')

n_valid = np.mean(lg.N, axis=0).astype(int)  # pixels valides par ordre

fig, axes = plot_logl_per_order(
    ccf_orders, lg.vsys_axis, lg.kp_axis,
    n_valid=n_valid,
    n_col=6,
    shared_clim=True,
)

## 5b. Contribution par ordre (leave-one-out, fraction hors-pic)

La contribution de l'ordre k mesure de combien il concentre le posterior vers le pic :

$$\text{contribution}_k = f_{\text{hors-pic}}^{\text{LOO}_k} - f_{\text{hors-pic}}^{\text{full}}$$

$$f_{\text{hors-pic}} = \frac{\displaystyle\sum_{\text{hors-pic}} P(v_\text{sys}, K_P)}{\displaystyle\sum_{\text{tout}} P(v_\text{sys}, K_P)}$$

La **zone hors-pic** est définie par les coins de la grille (loin du signal en vsys ET en Kp
simultanément), contrôlés par `vsys_excl` et `kp_excl`.

- **Barres bleues** : retirer l'ordre fait monter f_off → l'ordre concentrait la probabilité dans le pic → **bonne contribution**
- **Barres oranges** : retirer l'ordre fait baisser f_off → l'ordre étalait la probabilité → **mauvaise contribution**
- **Axe Y (fraction)** : part du signal total (somme des barres bleues = 1)

Cette métrique est insensible au scaling en N et ne dépend pas de l'emplacement exact du pic.

In [ ]:
posterior_full_loo, log_delta, contributions, contributions_frac, idx_ord_used = \
    lg.compute_loo_order_contributions(
        alpha_array=alpha_array,
        idx_signal=idx_signal,
        # idx_orders=[14, 15, 30, 31],  # restreindre à un sous-ensemble
        kind='BL',
        # kp_ref=kp_ref,          # centre de la zone d'exclusion (défaut: max de la carte)
        # vsys_ref=rv_expected,
        vsys_excl=30.,              # demi-largeur de la zone d'exclusion en vsys (km/s)
        kp_excl=30.,                # demi-largeur de la zone d'exclusion en Kp (km/s)
    )
print(f'{len(contributions)} ordres analysés')
best_k = np.argmax(contributions_frac)
worst_k = np.argmin(contributions_frac)
print(f'Ordre le plus contributif : {idx_ord_used[best_k]}  '
      f'({contributions_frac[best_k]*100:.1f} % du signal)')
print(f'Ordre le plus nuisible    : {idx_ord_used[worst_k]}  '
      f'({contributions_frac[worst_k]*100:.1f} % du signal)')

In [ ]:
%matplotlib inline

fig, (ax_bar, axes_maps) = plot_loo_contributions(
    posterior_full_loo, log_delta, contributions,
    vsys_axis=lg.vsys_axis,
    kp_axis=lg.kp_axis,
    idx_orders=idx_ord_used,
    contributions_frac=contributions_frac,  # affiche la fraction plutôt que Δlog P
    n_col=6,
    # kp_ref=kp_ref,          # marque la position de référence sur les cartes
    # vsys_ref=rv_expected,
    # vsys_lim=(-30, 30),     # zoom sur la région de détection
    # kp_lim=(150, 300),
)
# fig.savefig('loo_order_contributions.pdf', bbox_inches='tight')

## 6. Cartes alpha–Kp et alpha–vsys

Montrent la distribution jointe entre l'amplitude du modèle α et chaque paramètre
cinématique. Une détection produit un pic près de α ≈ 1.

In [ ]:
post_akp, alpha_os_akp, kp_os_akp, marg_alpha_akp, marg_kp_akp = \
    lg.compute_alpha_kp_posterior(alpha_array=alpha_array, idx_signal=idx_signal, oversample=2)

fig_akp, axes_akp = plot_alpha_rv_map(
    post_akp, alpha_os_akp, kp_os_akp, marg_alpha_akp, marg_kp_akp,
    rv_label=r'$K_{\rm P}$ (km s$^{-1}$)',
    n_sigma=3,
)
axes_akp[0].axhline(kp_ref, color='r', linestyle='--', alpha=0.5, label=r'$K_P$ ref')
axes_akp[0].legend(fontsize=9)

In [ ]:
post_avs, alpha_os_avs, vsys_os_avs, marg_alpha_avs, marg_vsys_avs = \
    lg.compute_alpha_vsys_posterior(alpha_array=alpha_array, idx_signal=idx_signal, oversample=2)

fig_avs, axes_avs = plot_alpha_rv_map(
    post_avs, alpha_os_avs, vsys_os_avs, marg_alpha_avs, marg_vsys_avs,
    rv_label=r'$v_{\rm sys}$ (km s$^{-1}$)',
    n_sigma=3,
)
axes_avs[0].axhline(rv_expected, color='r', linestyle='--', alpha=0.5, label=r'$v_{\rm sys}$ ref')
axes_avs[0].legend(fontsize=9)

## 7. Sauvegarde des posteriors

Sauvegarde le posterior combiné + axes pour réutilisation dans `model_comparison.ipynb`.

In [ ]:
save_posterior = True  # mettre à False pour désactiver

if save_posterior:
    # Identifiant du modèle : stem du premier fichier jusqu'au '_kp'
    import re
    model_id = re.sub(r'_kp.*', '', files[0].stem)

    save_dir = path_results / 'posteriors'
    save_dir.mkdir(parents=True, exist_ok=True)
    save_file = save_dir / f'{model_id}_posterior.npz'

    np.savez(
        save_file,
        posterior=posterior,
        vsys_axis=vsys_os,
        kp_axis=kp_os,
        margin_vsys=margin_vsys,
        margin_kp=margin_kp,
        model_id=model_id,
    )
    print(f'Sauvegardé : {save_file}')